# Loading Dataset

In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
epochs = 1
beta = 0.1
batch_size = 1
grad_accumulation_steps = 4

In [2]:
dataset = load_dataset("argilla/ultrafeedback-binarized-preferences-cleaned")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['source', 'prompt', 'chosen', 'chosen-rating', 'chosen-model', 'rejected', 'rejected-rating', 'rejected-model'],
        num_rows: 60917
    })
})

In [4]:
dataset = dataset["train"].select([i for i in range(1000)])

In [5]:
dataset = dataset.remove_columns(
    [
        "prompt",
        "source",
        "chosen-rating",
        "chosen-model",
        "rejected-rating",
        "rejected-model",
    ]
)

In [6]:
dataset[0]

{'chosen': [{'content': 'Can you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here\'s some starter code to help you out:\n#include <iostream>\n#include <string>\nusing namespace std;\nint main() {\n    string country;\n    // prompt user for input\n    cout << "Enter the name of a country: ";\n    cin >> country;\n    // check if country borders the Mediterranean Sea\n    // [C++ code]\n    return 0;\n}',
   'role': 'user'},
  {'content': 'Here\'s a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea:\n\n#include <iostream>\n#include <string>\n#include <set>\n#include <map>\n#include <algorithm>\n\nusing namespace std;\n\nint main() {\n    // store countries and their bordering seas in a map\n    map<string, set<string>> countries;\n    countries["Algeria"] = {"Mediterranean Sea", "North African Coast"};\n    countries["France"] = {"Mediterranean Sea",

# Loading Model

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.nn.utils.rnn import pad_sequence

In [8]:
model = AutoModelForCausalLM.from_pretrained(
    "Pankaj121212/sft_fin_llm", trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(
    "Pankaj121212/PM_MiniFinLLM", trust_remote_code=True
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

PM_MiniFinLLM_Model LOAD REPORT from: Pankaj121212/sft_fin_llm
Key                 | Status  | 
--------------------+---------+-
model.linear.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
print(
    tokenizer.decode(tokenizer.apply_chat_template(dataset[0]["chosen"])["input_ids"])
)

[BOS] Can you write a C + + program that promp ts the user to enter the name of a country and checks if it borders the Mediterranean Sea ? Here ' s some star ter code to help you out : # include < io stream > # include < string > using nam esp ace st d ; int main ( ) { string country ; // prompt user for input co ut < < " Enter the name of a country : " ; cin > > country ; // check if country borders the Mediterranean Sea // [ C + + code ] return 0 ; } [CLS] [SEP] Here ' s a C + + program that promp ts the user to enter the name of a country and checks if it borders the Mediterranean Sea : # include < io stream > # include < string > # include < set > # include < map > # include < algorithm > using nam esp ace st d ; int main ( ) { // store countries and their border ing se as in a map map < string , set < string > > countries ; countries [ " Al ger ia " ] = { " Mediterranean Sea ", " North African Coast " } ; countries [ " France " ] = { " Mediterranean Sea ", " English Channel " } ; 

In [10]:
print(
    tokenizer.decode(tokenizer.apply_chat_template(dataset[0]["rejected"])["input_ids"])
)

[BOS] Can you write a C + + program that promp ts the user to enter the name of a country and checks if it borders the Mediterranean Sea ? Here ' s some star ter code to help you out : # include < io stream > # include < string > using nam esp ace st d ; int main ( ) { string country ; // prompt user for input co ut < < " Enter the name of a country : " ; cin > > country ; // check if country borders the Mediterranean Sea // [ C + + code ] return 0 ; } [CLS] [SEP] Sure , here is the program using the C + + 11 algorithm " c ds : : algorithm : : Greater Equal " : # include < io stream > # include < string > # include < algorithm > # include < vector > # include < c ct y pe > using nam esp ace st d ; int main ( ) { string country ; co ut < < " Enter the name of a country : " ; cin > > country ; st d : : vector < string > v ec ; v ec . push _ back ( country ); size _ t index = st d : : find _ if ( v ec . begin ( ), v ec . end ( ), [ ] ( con st string & s ) { return st d : : any _ of ( s . 

# Preprocessing

In [11]:
def preprocess_fn(inp_row):
    chosen_tokens = tokenizer.apply_chat_template(
        inp_row["chosen"], return_assistant_tokens_mask=True, return_tensors="pt"
    )
    rejected_tokens = tokenizer.apply_chat_template(
        inp_row["rejected"], return_assistant_tokens_mask=True, return_tensors="pt"
    )
    return {"chosen": chosen_tokens, "rejected": rejected_tokens}

In [12]:
dataset = dataset.map(preprocess_fn, batched=False)

In [13]:
print(dataset[0]["chosen"]["input_ids"][0])

[5, 3217, 3360, 4254, 70, 40, 16, 16, 2506, 1301, 19066, 1267, 1223, 4859, 1228, 2328, 1223, 4198, 1226, 70, 4396, 1227, 11343, 1344, 1239, 20227, 1223, 29885, 11292, 36, 30034, 12, 88, 2149, 5438, 1307, 6948, 1228, 3822, 3360, 1544, 31, 8, 1919, 33, 2344, 4592, 35, 8, 1919, 33, 5804, 35, 2405, 3902, 4515, 1866, 1242, 73, 32, 1529, 1692, 13, 14, 96, 5804, 4396, 32, 8431, 6935, 4859, 1253, 7865, 1237, 1305, 33, 33, 7, 5203, 1223, 4198, 1226, 70, 4396, 31, 7, 32, 11429, 35, 35, 4396, 32, 8431, 5906, 1344, 4396, 20227, 1223, 29885, 11292, 8431, 64, 40, 16, 16, 6948, 66, 2411, 21, 32, 98, 1, 2, 30034, 12, 88, 70, 40, 16, 16, 2506, 1301, 19066, 1267, 1223, 4859, 1228, 2328, 1223, 4198, 1226, 70, 4396, 1227, 11343, 1344, 1239, 20227, 1223, 29885, 11292, 31, 8, 1919, 33, 2344, 4592, 35, 8, 1919, 33, 5804, 35, 8, 1919, 33, 1739, 35, 8, 1919, 33, 11608, 35, 8, 1919, 33, 21430, 35, 2405, 3902, 4515, 1866, 1242, 73, 32, 1529, 1692, 13, 14, 96, 8431, 3604, 2869, 1227, 1622, 10758, 1231, 1314, 1232

In [14]:
tokenizer.pad_token_id

3

In [15]:
def data_collator(batch):
    pad_token = tokenizer.pad_token_id

    batch_chosen_masks = [
        torch.tensor(i["chosen"]["assistant_masks"][0]) for i in batch
    ]
    batch_rejected_masks = [
        torch.tensor(i["rejected"]["assistant_masks"][0]) for i in batch
    ]
    batch_chosen_ids = [torch.tensor(i["chosen"]["input_ids"][0]) for i in batch]
    batch_rejected_ids = [torch.tensor(i["rejected"]["input_ids"][0]) for i in batch]

    batch_chosen_masks = pad_sequence(
        batch_chosen_masks, batch_first=True, padding_value=0
    )
    batch_rejected_masks = pad_sequence(
        batch_rejected_masks, batch_first=True, padding_value=0
    )
    batch_chosen_ids = pad_sequence(
        batch_chosen_ids, batch_first=True, padding_value=pad_token
    )
    batch_rejected_ids = pad_sequence(
        batch_rejected_ids, batch_first=True, padding_value=pad_token
    )

    return {
        "chosen_ids": batch_chosen_ids,
        "chosen_masks": batch_chosen_masks,
        "rejected_ids": batch_rejected_ids,
        "rejected_masks": batch_rejected_masks,
    }

In [16]:
dataloader = DataLoader(dataset, collate_fn=data_collator, batch_size=batch_size)

In [17]:
x = next(iter(dataloader))
x["chosen_ids"].shape, x["chosen_masks"].shape, x["rejected_ids"].shape, x[
    "rejected_masks"
].shape

(torch.Size([1, 636]),
 torch.Size([1, 636]),
 torch.Size([1, 386]),
 torch.Size([1, 386]))

# Training

In [18]:
def get_seq_log_probs(model, inp, mask, device=device):

    logits = model(inp).logits

    shifted_logits = logits[:, :-1, :]
    shifted_labels = inp[:, 1:]
    mask = mask[:, 1:]

    logps = nn.functional.log_softmax(shifted_logits, dim=-1)

    per_token_logps = logps.gather(dim=-1, index=shifted_labels.unsqueeze(-1)).squeeze(
        -1
    )
    per_token_logps = per_token_logps * mask

    return per_token_logps.sum(dim=1)

In [19]:
import copy

reference_model = copy.deepcopy(model)
policy_model = model

In [20]:
for param in reference_model.parameters():
    param.requires_grad = False

In [21]:
optimizer = torch.optim.Adam(
    policy_model.parameters(), lr=1e-6, betas=(0.9, 0.999), weight_decay=0.01
)

In [23]:
reference_model = reference_model.to(device)
policy_model = policy_model.to(device)

In [24]:
len(dataloader)

1000

In [25]:
for epoch in range(epochs):
    print("EPOCH", epoch + 1)
    p_bar = tqdm(total=len(dataloader), desc="training")
    loss = 0
    for i, batch in enumerate(dataloader):
        batch_chosen_ids = batch["chosen_ids"].to(device)
        batch_chosen_masks = batch["chosen_masks"].to(device)
        batch_rejected_ids = batch["rejected_ids"].to(device)
        batch_rejected_masks = batch["rejected_masks"].to(device)

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            policy_chosen = get_seq_log_probs(
                policy_model, batch_chosen_ids, batch_chosen_masks
            )
            policy_rejected = get_seq_log_probs(
                policy_model, batch_rejected_ids, batch_rejected_masks
            )

            with torch.no_grad():
                reference_chosen = get_seq_log_probs(
                    reference_model, batch_chosen_ids, batch_chosen_masks
                )
                reference_rejected = get_seq_log_probs(
                    reference_model, batch_rejected_ids, batch_rejected_masks
                )

            raw_lss = -nn.functional.logsigmoid(
                beta
                * (
                    policy_chosen
                    - reference_chosen
                    - (policy_rejected - reference_rejected)
                )
            ).mean()
            loss += raw_lss

            if i % grad_accumulation_steps == 0:
                optimizer.zero_grad()
                loss = loss / grad_accumulation_steps
                loss.backward()

                loss = 0
                torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
                optimizer.step()

        p_bar.set_postfix(loss=raw_lss.item())
        p_bar.update(1)

EPOCH 1


training: 100%|██████████| 1000/1000 [09:33<00:00,  1.41it/s, loss=0.809]

In [ ]:
!hf auth login --token your_hf_token

Hint: A new version of huggingface_hub (1.16.4) is available! You are using version 1.16.1.
To update, run: hf update
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `MyLLm` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `MyLLm`


In [27]:
policy_model.push_to_hub("Pankaj121212/DPO_finLLM")

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0yhos74/model.safetensors:   0%|          | 14.5kB /  439MB            

CommitInfo(commit_url='https://huggingface.co/Pankaj121212/DPO_finLLM/commit/94c10721304d1d53503989aaf8764121ac86e9b0', commit_message='Upload model', commit_description='', oid='94c10721304d1d53503989aaf8764121ac86e9b0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Pankaj121212/DPO_finLLM', endpoint='https://huggingface.co', repo_type='model', repo_id='Pankaj121212/DPO_finLLM'), pr_revision=None, pr_num=None)